# CNN dish classifier workflow

Colab workflow for preparing, training, evaluating, and exporting the single PyTorch dish classifier used by the checkout demo.


## 1. Configure the repository and Drive


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
REPO = "Vo-Minh-Tri1412/cnn-food-recognition"
BRANCH = "main"
PROJECT_ROOT = Path("/content/cnn-food-recognition")
DRIVE_ROOT = Path("/content/drive/MyDrive/cnn-food-recognition")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import subprocess

os.chdir("/content")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


## 2. Prepare the classification dataset


In [ ]:
import subprocess

subprocess.run(["python", "scripts/data/01_build_classification_dataset.py", "--clear"], check=True)
subprocess.run(["python", "scripts/data/02_audit_dataset_conflicts.py", "--root", "data/classification", "--phash-threshold", "4"], check=True)
subprocess.run(["python", "scripts/data/03_package_classification_dataset.py"], check=True)


## 3. Train the CNN classifier


In [ ]:
from datetime import datetime
import subprocess

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = DRIVE_ROOT / "runs" / "classifier" / RUN_ID
MODEL_OUT = DRIVE_ROOT / "models" / "dish_classifier.pt"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)

subprocess.run([
    "python", "scripts/train/01_train_classifier.py",
    "--data", "data/classification",
    "--epochs", "20",
    "--batch-size", "32",
    "--patience", "3",
    "--model-out", str(MODEL_OUT),
    "--output-dir", str(RUN_ROOT),
], check=True)


## 4. Evaluate and export


In [ ]:
import json
import torch

checkpoint = torch.load(MODEL_OUT, map_location="cpu", weights_only=False)
summary = {
    "model": str(MODEL_OUT),
    "architecture": checkpoint.get("arch"),
    "classes": checkpoint.get("class_names", []),
    "image_size": checkpoint.get("image_size"),
}
(RUN_ROOT / "export_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary
